# Notebook 02 - Isolation Forest + LOSO Cross-Validation

Train model bat thuong da chi so tren `health_data_augmented.csv`, danh gia Leave-One-Subject-Out va luu `isolation_forest.pkl`, `scaler.pkl`.

In [ ]:
# Cell 1
from pathlib import Path
import shutil

import pandas as pd, numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score
from sklearn.preprocessing import MinMaxScaler
import joblib, json

DATA_CANDIDATES = [
    Path('health_data_augmented.csv'),
    Path('../colab_export/health_data_augmented.csv'),
    Path('ai_training/colab_export/health_data_augmented.csv'),
    Path('/content/health_data_augmented.csv'),
]
data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Khong tim thay health_data_augmented.csv. Hay upload file len Colab truoc.')

df = pd.read_csv(data_path)
df['label'] = df['label'].astype(str).str.strip().str.lower()
features = ['heart_rate','spo2','temperature','hr_rolling_mean',
            'hr_rolling_std','spo2_change','accel_mag']
print('Loaded:', data_path)
print('Rows:', len(df))
print('Subjects:', df['subject_id'].nunique())
print('\nLabel distribution:')
print(df['label'].value_counts())

In [ ]:
# Cell 2: LOSO Cross-Validation
subjects = df['subject_id'].unique()
all_f1 = []

for test_sub in subjects:
    train_df = df[df['subject_id'] != test_sub]
    test_df  = df[df['subject_id'] == test_sub]
    train_normal = train_df[train_df['label'].str.startswith('normal', na=False)]

    X_train = train_normal[features].dropna()
    if X_train.empty:
        print(f'  Exclude {test_sub}: SKIP - khong co mau normal trong train_df')
        continue

    scaler = MinMaxScaler().fit(X_train)
    model = IsolationForest(contamination=0.05, n_estimators=200, random_state=42)
    model.fit(scaler.transform(X_train))

    X_test = scaler.transform(test_df[features].fillna(0))
    preds  = model.predict(X_test)
    y_true = (test_df['label'].str.startswith('anomaly', na=False) | (test_df['label'] == 'fall')).astype(int)
    y_pred = (preds == -1).astype(int)

    f1 = f1_score(y_true, y_pred, zero_division=0)
    all_f1.append(f1)
    print(f'  Exclude {test_sub}: F1={f1:.3f}')

if all_f1:
    print(f'LOSO F1 Mean: {np.mean(all_f1):.3f} ± {np.std(all_f1):.3f}')
else:
    raise ValueError('Khong co fold LOSO nao train duoc. Kiem tra label: can co nhan bat dau bang normal, vi du normal_rest.')

In [ ]:
# Cell 3: Train model global
all_normal = df[df['label'].str.startswith('normal', na=False)][features].dropna()
if all_normal.empty:
    raise ValueError('Khong co mau normal de train model global. Kiem tra cot label trong health_data_augmented.csv')
scaler_g = MinMaxScaler()
model_g  = IsolationForest(contamination=0.05, n_estimators=300, random_state=42)
model_g.fit(scaler_g.fit_transform(all_normal))

joblib.dump(model_g, 'isolation_forest.pkl')
joblib.dump(scaler_g, 'scaler.pkl')

model_info = {
    'model': 'IsolationForest',
    'features': features,
    'contamination': 0.05,
    'n_estimators': 300,
    'n_samples_normal': int(len(all_normal)),
    'loso_f1_mean': float(np.mean(all_f1)),
    'loso_f1_std': float(np.std(all_f1)),
}
with open('model_info.json', 'w', encoding='utf-8') as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False)

print(f'Model luu OK! LOSO F1={np.mean(all_f1):.3f}')

In [ ]:
# Cell 4: Test cases cu the
test_cases = [
    {'name':'Binh thuong',   'data':[70, 98, 36.5, 70, 2,  0,  1.0]},
    {'name':'Nhip tim cao',  'data':[155,97, 36.8,145, 8, -1,  1.1]},
    {'name':'SpO2 thap',     'data':[88, 87, 36.5, 88, 3, -2,  1.0]},
    {'name':'Sot cao',       'data':[100,96, 39.5, 98, 4,  0,  1.0]},
    {'name':'Phat hien nga', 'data':[90, 97, 36.8, 88, 5,  0,  3.2]},
]

print('\n=== KET QUA TEST AI ===')
for tc in test_cases:
    X_t  = scaler_g.transform([tc['data']])
    pred = model_g.predict(X_t)[0]
    sc   = model_g.decision_function(X_t)[0]
    status = 'Binh thuong' if pred == 1 else 'Bat thuong'
    print(f"{tc['name']:20s}: {status} (score={sc:.3f})")

In [ ]:
# Cell 5: Download va copy vao models/ neu co project trong Colab
try:
    from google.colab import files
    files.download('isolation_forest.pkl')
    files.download('scaler.pkl')
    files.download('model_info.json')
except Exception as exc:
    print('Khong phai Colab hoac khong download duoc:', exc)

candidate_model_dirs = [
    Path('/content/aiot_health/models'),
    Path('/content/drive/MyDrive/aiot_health/models'),
    Path('../../models'),
    Path('models'),
]
for model_dir in candidate_model_dirs:
    if model_dir.exists():
        for name in ['isolation_forest.pkl', 'scaler.pkl', 'model_info.json']:
            shutil.copy2(name, model_dir / name)
        print(f'Copied artifacts to: {model_dir.resolve()}')
        break